# Tutorial 2: Fine-tuning

ElemeNet supports two distinct methods of fine-tuning (i.e., continuing training from existing model `.pt` checkpoint files).

Resuming a trained model restores the weights and optimizer state, essentially continuing a previously interrupted run for additional epochs. This is useful if examining your training and validation loss curves reveals that validation loss has not yet plateaued, and the model has additional learning capacity.

Tranfer learning restores only the weights and uses a fresh optimizer. This provides a means to initialize a new model from the pretrained weights of an existing model. This initialization scheme is widely used for related learning tasks, especially when the pretrained model was developed on a larger dataset, as in the case of so-called "foundation models".

`elemenet_train` provides access to both methods:

| | Restores | Use-case | CLI flag |
|---|---|---|---|
| **Resume** | model weights and optimizer state (LR, momentum, epoch counter) | continue an interrupted run, or train longer | `--resume <run_dir or checkpoint>` |
| **Transfer-learn** | model weights only (fresh optimizer) | adapt a pretrained model to a new target or dataset | `--transfer_learn <checkpoint>` |

## Option A: Resuming a run (weights and optimizer)

Point `--resume` at either the run directory or a specific checkpoint file. The CLI infers `save_dir` and the optimizer state automatically, and reuses the run's own `config.yaml`. Remember to increase `epochs` in the config before resuming.

**Note:** If you used a `cosine` or `onecycle` learning-rate schedule, resume with the same `epochs` you originally trained with, as these schedulers are tied to the total epoch count. To extend training with a fresh schedule, transfer-learn with `--transfer_learn` instead (next section). ElemeNet will warn you if it detects a mismatch.

In [ ]:
# Resume from a run directory (uses checkpoints/last.pt inside it):
!elemenet_train --config model/config.yaml --resume model

# Resume from a specific checkpoint file:
!elemenet_train --config model/config.yaml --resume model/checkpoints/last.pt

## Option B: Transfer-learning (weights only)

To initialize a new run from pretrained weights (typically against a different dataset or target), use `--transfer_learn`. The optimizer is restarted from scratch, so you should point `--config` at a new config (new data, new `save_dir`, possibly a new `target_column`).

In [ ]:
!elemenet_train --config transfer_config.yaml --transfer_learn model/checkpoints/best.pt

`--resume` and `--transfer_learn` are mutually exclusive — the CLI will error if you pass both.

## SLURM example

```bash
#!/bin/bash
#SBATCH --job-name=elemenet_finetune
#SBATCH -o training_test.out
#SBATCH --gres=gpu:1
#SBATCH --mem=64G
#SBATCH -c 4

conda activate ElemeNet

elemenet_train --config config.yaml --resume model/checkpoints/last.pt
```